In [4]:
import pandas as pd
from yfetch import get_stock_history
symbols = ['ESIF.DE','NUKL.DE','DFEN.DE','VDIV.DE']
rows = []
for symbol in symbols:
    history = get_stock_history(symbol, period='2y', interval='1d')
    monthly = history.resample('ME').agg({'High': 'max', 'Low': 'min'})
    monthly['Swing'] = 2 * (monthly.High - monthly.Low) / (monthly.High + monthly.Low)
    avg_swing = monthly.Swing.mean()
    last_price = history.Close.iloc[-1]
    one_year_ago = history.index[-1] - pd.DateOffset(years=1)
    past_year = history[history.index >= one_year_ago]
    prev_year = history[(history.index < one_year_ago)]
    avg_past_year = past_year.Close.mean()
    avg_prev_year = prev_year.Close.mean()
    yoy_change = avg_past_year / avg_prev_year - 1
    rows.append({
        'Symbol': symbol,
        'Price': last_price,
        'Swing': avg_swing,
        'YoY Avg': yoy_change,
        'R/R': yoy_change / avg_swing if avg_swing > 0 else float('inf')
    })

results = pd.DataFrame(rows).set_index('Symbol')
results['Price'] = results['Price'].map('{:.2f}'.format)
for col in ['Swing', 'YoY Avg']:
    results[col] = results[col].map('{:.1%}'.format)
results

Fetched history for VDIV.DE (505 rows)


,Price,Swing,YoY Avg,R/R
Symbol,,,,
ESIF.DE,14.84,7.4%,35.5%,4.807843
NUKL.DE,54.62,18.0%,64.5%,3.589905
DFEN.DE,53.70,10.3%,56.9%,5.528050
VDIV.DE,51.77,4.9%,20.7%,4.224705
